# 配置组装与 HiPNUC IMU

从 YAML 初始化设备，并通过 graphmap Pose 查看参考外参。默认只打开新 IMU，不连接底盘。将 `MODE` 改成 `ros2` 可切换到已构建的 ROS2 适配包；两种方式串行运行，不能独占打开同一串口。

IMU 平移及安装朝向尚未标定，读取数据不依赖这些外参。相机/雷达沿用参考配置的几何信息。

In [ ]:
import os
from pathlib import Path
import sys
import numpy as np
from rsim import Runtime
from rsim.config import load_rig, UncalibratedMount

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "rsim").is_dir() and (p / "pyproject.toml").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from examples.imu_rig import capture

ASSETS = ROOT / "assets" / "hipnuc"
ASSETS.mkdir(parents=True, exist_ok=True)
os.environ["NOTEBOOK_ASSETS_ROOT"] = str(ROOT / "assets")
os.environ["NOTEBOOK_NAME"] = "hipnuc"
from scipykit.mtp_initializer import subplots, label_axes, disp, plt

MODE = "serial"  # "serial" or "ros2"
rig = load_rig(ROOT / "configs" / "sensors.yaml", select="imu",
               overrides={"imu": {"mode": MODE, "log_path": str(ASSETS / "notebook-driver.log")}})
print("IMU mount:", rig["imu"].mount)

In [ ]:
async with Runtime(rig):
    report, frames = await capture(rig["imu"].imu, 200, 15)
print({key: value for key, value in report.items() if key != "last"})
print("Last IMU sample:", report["last"])
values = np.array([[frame.stamp_ns * 1e-9,
                    *[frame.data["linear_acceleration"][axis] for axis in "xyz"],
                    *[frame.data["angular_velocity"][axis] for axis in "xyz"]]
                   for frame in frames])
np.save(ASSETS / f"notebook-{MODE}.npy", values)
fig, axes = subplots(1, 2, figsize=(10, 3.3), layout="constrained")
for i, axis in enumerate("xyz"):
    axes[0].plot(values[:, 0] - values[0, 0], values[:, i+1], label=axis)
    axes[1].plot(values[:, 0] - values[0, 0], values[:, i+4], label=axis)
label_axes(axes[0], "reception time (s)", "acceleration (m/s²)", grid=True, legend=True)
label_axes(axes[1], "reception time (s)", "angular velocity (rad/s)", grid=True, legend=True)
disp(fig, f"{MODE}-imu")
plt.close(fig)

## 固定外参与可复用组合

下面只构建对象、读取外参，不进入 Runtime，所以不会启动相机/雷达。需要联合采集时可使用 `async with Runtime(assembly)`；各传感器保留自己的时间域，这不等于完成时间同步。

In [ ]:
assembly = load_rig(ROOT / "configs" / "sensors.yaml", select="robot")
assert assembly["robot"]["perception"]["imu"] is assembly["inertial"]["imu"]
T_rgb_lidar3d = assembly.transform("rgb", "lidar3d")
print("T_rgb_lidar3d:", T_rgb_lidar3d)
print("LiDAR origin in camera FLU (m):", T_rgb_lidar3d([0., 0., 0.]))
try:
    assembly.transform("rgb", "imu")
except UncalibratedMount as error:
    print("Expected until calibration:", error)

原始加速度保留重力，协方差全零表示设备未提供。旧协议没有设备采样时间，图中为接收时间。坐标轴和安装外参需要单独核实，不把 IMU 的导航系当作 odom 或机器人基座。更多说明见 [IMU 接入](../docs/imu.md) 与 [配置组装](../docs/configuration.md)。